In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import DataStructs, rdMolDescriptors, rdReducedGraphs
from rdkit.Avalon.pyAvalonTools import GetAvalonCountFP
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
from mordred import Calculator, descriptors as mordred_descriptors

# Descriptors Selected
In this project, we have selected a set of molecular descriptors that are commonly used in toxicity prediction models. 

We selected features that are relevant to the toxicity of chemical compounds, taken from different papers in the field. These descriptors include:
- [General Descriptors](https://github.com/partex-nv-opensource/tdc-submission/blob/main/src/ics-admet-prediction-tdc-versiontwo/report/Advancing%20ADMET%20Prediction_%20A%20Hybrid%20Machine%20Learning%20Framework%20(Partex%20ADMETrix).pdf): basic molecular properties such as molecular weight, logP, number of rotatable bonds, etc.
- [Dragon descriptors](https://pubs.acs.org/doi/abs/10.1021/tx900189p?casa_token=vfBbuxuUCqEAAAAA:YAcI0r4Z3rtlRYP_l5H8OlTfdUh3DVlO6ws_h1NkhpaXH3-NrdI2-s5ghWWJbxfPQw-KhQIAwMi1Di3v): more complex descriptors that capture various aspects of molecular structure and properties, such as topological indices, geometrical descriptors, and electronic properties.

In [ ]:
DESCRIPTORS = [
        'BalabanJ', 'BertzCT', 'Chi0', 'Chi0n', 'Chi0v', 'Chi1', 
        'Chi1n', 'Chi1v', 'Chi2n', 'Chi2v', 'Chi3n', 'Chi3v', 'Chi4n', 'Chi4v', 
        'EState_VSA1', 'EState_VSA10', 'EState_VSA11', 'EState_VSA2', 'EState_VSA3', 
        'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 
        'EState_VSA9', 'ExactMolWt', 'FpDensityMorgan1', 'FpDensityMorgan2', 
        'FpDensityMorgan3', 'FractionCSP3', 'HallKierAlpha', 'HeavyAtomCount', 
        'HeavyAtomMolWt', 'Ipc', 'Kappa1', 'Kappa2', 'Kappa3', 'LabuteASA', 
        'MaxAbsEStateIndex', 'MaxAbsPartialCharge', 'MaxEStateIndex', 'MaxPartialCharge', 
        'MinAbsEStateIndex', 'MinAbsPartialCharge', 'MinEStateIndex', 'MinPartialCharge', 
        'MolLogP', 'MolMR', 'MolWt', 'NHOHCount', 'NOCount', 'NumAliphaticCarbocycles', 
        'NumAliphaticHeterocycles', 'NumAliphaticRings', 'NumAromaticCarbocycles', 
        'NumAromaticHeterocycles', 'NumAromaticRings', 'NumHAcceptors', 'NumHDonors', 
        'NumHeteroatoms', 'NumRadicalElectrons', 'NumRotatableBonds', 
        'NumSaturatedCarbocycles', 'NumSaturatedHeterocycles', 'NumSaturatedRings', 
        'NumValenceElectrons', 'PEOE_VSA1', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 
        'PEOE_VSA13', 'PEOE_VSA14', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 
        'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'RingCount', 'SMR_VSA1', 
        'SMR_VSA10', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 
        'SMR_VSA8', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA10', 'SlogP_VSA11', 'SlogP_VSA12', 
        'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6', 'SlogP_VSA7', 
        'SlogP_VSA8', 'SlogP_VSA9', 'TPSA', 'VSA_EState1', 'VSA_EState10', 'VSA_EState2', 
        'VSA_EState3', 'VSA_EState4', 'VSA_EState5', 'VSA_EState6', 'VSA_EState7', 
        'VSA_EState8', 'VSA_EState9', 'fr_Al_COO', 'fr_Al_OH', 'fr_Al_OH_noTert', 'fr_ArN', 
        'fr_Ar_COO', 'fr_Ar_N', 'fr_Ar_NH', 'fr_Ar_OH', 'fr_COO', 'fr_COO2', 'fr_C_O', 
        'fr_C_O_noCOO', 'fr_C_S', 'fr_HOCCN', 'fr_Imine', 'fr_NH0', 'fr_NH1', 'fr_NH2', 
        'fr_N_O', 'fr_Ndealkylation1', 'fr_Ndealkylation2', 'fr_Nhpyrrole', 'fr_SH', 
        'fr_aldehyde', 'fr_alkyl_carbamate', 'fr_alkyl_halide', 'fr_allylic_oxid', 
        'fr_amide', 'fr_amidine', 'fr_aniline', 'fr_aryl_methyl', 'fr_azide', 'fr_azo', 
        'fr_barbitur', 'fr_benzene', 'fr_benzodiazepine', 'fr_bicyclic', 'fr_diazo', 
        'fr_dihydropyridine', 'fr_epoxide', 'fr_ester', 'fr_ether', 'fr_furan', 'fr_guanido', 
        'fr_halogen', 'fr_hdrzine', 'fr_hdrzone', 'fr_imidazole', 'fr_imide', 'fr_isocyan', 
        'fr_isothiocyan', 'fr_ketone', 'fr_ketone_Topliss', 'fr_lactam', 'fr_lactone', 
        'fr_methoxy', 'fr_morpholine', 'fr_nitrile', 'fr_nitro', 'fr_nitro_arom', 
        'fr_nitro_arom_nonortho', 'fr_nitroso', 'fr_oxazole', 'fr_oxime', 
        'fr_para_hydroxylation', 'fr_phenol', 'fr_phenol_noOrthoHbond', 'fr_phos_acid', 
        'fr_phos_ester', 'fr_piperdine', 'fr_piperzine', 'fr_priamide', 'fr_prisulfonamd', 
        'fr_pyridine', 'fr_quatN', 'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 
        'fr_term_acetylene', 'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene', 
        'fr_unbrch_alkane', 'fr_urea', 'qed'
    ]

def get_descriptors_str():
    return DESCRIPTORS

def compute_rdkit_features(mols):
    """Compute RDKit descriptors for a series of molecules."""
    descriptor_calculator = MolecularDescriptorCalculator(get_descriptors_str())
    rdkit_features = mols.apply(lambda x: np.array(descriptor_calculator.CalcDescriptors(x)))
    return np.vstack(rdkit_features.values)

In [ ]:
def compute_dragon_like_features(mols, ignore_3d=True):
    """
    Computes Dragon-like descriptors using the Mordred library.
    ignore_3d=True is recommended for 2D SMILES datasets.
    """
    # all 2D descriptors
    calc = Calculator(mordred_descriptors, ignore_3d=ignore_3d)
    
    # compute descriptors for the list of RDKit molecules
    # quiet=True removes the progress bar for cleaner notebook output
    df_mordred = calc.pandas(mols, quiet=False)
    
    # handle non-numeric errors (Mordred returns error objects for failed molecules)
    # We convert errors to NaN and fill with 0 to ensure the matrix is usable
    df_numeric = df_mordred.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    return df_numeric.values

## Fingerprints creation
We will create three types of fingerprints: Morgan, Avalon, and ErG. These fingerprints will be generated using the RDKit library and will be used as features for our toxicity prediction model.

Reference: [Advancing ADMET Prediction: A Hybrid Machine Learning Framework (Partex ADMETrix)](https://github.com/partex-nv-opensource/tdc-submission/blob/main/src/ics-admet-prediction-tdc-versiontwo/report/Advancing%20ADMET%20Prediction_%20A%20Hybrid%20Machine%20Learning%20Framework%20(Partex%20ADMETrix).pdf)

In [ ]:
def convert_to_array(fp):
    arr = np.zeros((0,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def generate_avalon_fps(mols, n_bits=1024):
    fps = mols.apply(lambda x: GetAvalonCountFP(x, nBits=n_bits))
    # return np.stack(fps.apply(convert_to_array).values)
    try:
        return np.stack(fps.apply(convert_to_array).values)
    except Exception as e:
        print(f"Error generating Avalon fingerprints: {e}")
        return None

def generate_morgan_fps(mols, n_bits=1024, radius=2):
    fps = mols.apply(lambda x: rdMolDescriptors.GetHashedMorganFingerprint(x, nBits=n_bits, radius=radius))
    return np.stack(fps.apply(convert_to_array).values)

def generate_erg_fps(mols):
    fps = mols.apply(rdReducedGraphs.GetErGFingerprint)
    return np.stack(fps.values)

In [ ]:
def generate_all_features(df, name_smile_col='canonical_smiles'):
    """Combines original fingerprints with new Dragon-like descriptors."""
    df = df.copy()  # Avoid modifying the original DataFrame
    smiles_series = df[name_smile_col]
    mols = smiles_series.apply(Chem.MolFromSmiles)
    valid_mols = mols.dropna()
    
    feature_sets = [
        generate_morgan_fps(valid_mols),
        generate_avalon_fps(valid_mols),
        generate_erg_fps(valid_mols),
        compute_rdkit_features(valid_mols),
        compute_dragon_like_features(valid_mols) # Dragon-like set
    ]

    df['morgan_fp'] = list(feature_sets[0])
    df['avalon_fp'] = list(feature_sets[1])
    df['erg_fp'] = list(feature_sets[2])
    

    
    return np.concatenate(feature_sets, axis=1)

In [ ]:
def generate_all_features(df, mol_col='mol', quiet=False):
    """
    Computes fingerprints and descriptors, adding them as named columns directly to the DataFrame.
    Returns the modified DataFrame with all features attached.
    """
    df = df.copy()  # Avoid modifying the original DataFrame
    df = df.dropna(subset=[mol_col])  # Ensure we only process valid molecules
    
    # 1. Prepare Molecules
    # mols = df[mol_col]
    # valid_mols = mols.dropna()
    valid_mols = df[mol_col]

    # 2. Compute Fingerprints (Stored as list/array objects in single columns)
    print("Generating Fingerprints...")
    df['morgan_fp'] = list(generate_morgan_fps(valid_mols))
    df['avalon_fp'] = list(generate_avalon_fps(valid_mols))
    df['erg_fp'] = list(generate_erg_fps(valid_mols))

    # 3. Compute RDKit Descriptors (Expanded into individual columns)
    print("Computing RDKit Descriptors...")
    rdkit_feat_matrix = compute_rdkit_features(valid_mols)
    rdkit_names = get_descriptors_str() # From your descriptors.ipynb
    df_rdkit = pd.DataFrame(rdkit_feat_matrix, columns=rdkit_names, index=df.index)
    
    # 4. Compute Dragon-like (Mordred) Descriptors
    print("Computing Mordred Descriptors...")
    # We initialize the calculator to get the specific column names
    calc = Calculator(mordred_descriptors, ignore_3D=True)
    df_mordred = calc.pandas(valid_mols, quiet=quiet)
    
    # Clean Mordred output: convert error objects to NaN/0
    df_mordred = df_mordred.apply(pd.to_numeric, errors='coerce').fillna(0)
    # Add a prefix to Mordred columns to distinguish them from RDKit
    df_mordred.columns = [f"mordred_{c}" for c in df_mordred.columns]
    
    # 5. Merge all descriptors back to the main DataFrame
    df = pd.concat([df, df_rdkit, df_mordred], axis=1)
    
    print(f"Feature generation complete. Total columns: {len(df.columns)}")
    return df

In [ ]:
print("All functions defined. Ready to process datasets.")

All functions defined. Ready to process datasets.
